# 배민 외식배달비 간편 조사

아래 **설정 셀 한 곳만 수정**한 뒤 `미리보기 → 실제 조사` 순서로 실행하세요. 기존 745개 조사에 사용한 핵심 조사 함수는 `baemin_survey.py`에 들어 있습니다.

**MODE / TARGET 사용 예시**

| 조사 종류 | MODE | TARGET |
|---|---|---|
| 엑셀 486행 1개 | `"행"` | `[486]` |
| 엑셀 여러 행 | `"행"` | `[486, 487]` |
| 대상처 이름(중복시 행으로 선택) | `"대상처"` | `"가게 이름"` |
| 주소지 하나 | `"주소"` | `"엑셀 H열 주소 문자열"` |
| 지역 하나 | `"지역"` | `"세종"` |
| 지역 여러 개 | `"여러지역"` | `["대전", "세종"]` |
| 기존 조합 2, 4 | `"조합"` | `[2, 4]` |
| 전체 조사 | `"전체"` | `None` |

**기존 6개 조사 조합 (원본 조사표 745건 기준)**

| 조합 | 지역 |
|---|---|
| 1 | 경기 |
| 2 | 서울, 강원, 인천 |
| 3 | 경남, 전남, 제주 |
| 4 | 경북, 충북, 세종 |
| 5 | 부산, 광주, 대구, 울산 |
| 6 | 전북, 충남, 대전 |

**주의**

- `PREVIEW_ONLY = True`이면 원본/결과 파일을 수정하거나 BlueStacks에 연결하지 않습니다. 확인 후 `False`로 바꾸세요.
- 기존 조사표 월별 열 구조상 **9~12월만 실행하도록 안전 제한**했습니다. 다른 월은 Excel 열 배치를 검증한 뒤 지원해야 합니다.
- `NEW_SURVEY = True`는 초기화용입니다. 기존 결과가 있으면 확인 입력 및 백업을 수행합니다.
- 조합/전체는 `OUTPUT_FILE`에 `_자동전체조사.xlsx`를 붙인 별도 파일을 만들며 주소지마다 저장하고 이어하기를 지원합니다.
- 행/대상처 조사 시 같은 이름의 `*_자동전체조사.xlsx` 결과가 존재하면 그 결과를 우선 사용합니다.
- 실제 데이터 및 개인 경로를 입력한 실행 노트북은 **GitHub에 업로드하지 마세요.**
- `Kernel → Restart & Run All`은 실제 조사가 시작될 수 있으므로 주의하세요.

In [ ]:
from baemin_survey import run_survey

# 1. 조사 월과 Excel 경로
MONTH = 10
INPUT_FILE = r"C:\survey\input.xlsx"
OUTPUT_FILE = r"C:\survey\output.xlsx"

# 2. 조사 대상 (위 표 참고)
MODE = "행"
TARGET = [8]

# 3. 실행 설정
NEW_SURVEY = True
PREVIEW_ONLY = False
BLUESTACKS_DIR = r"C:\Program Files\BlueStacks_nxt"

In [ ]:
run_survey(
    month=MONTH,
    input_file=INPUT_FILE,
    output_file=OUTPUT_FILE,
    mode=MODE,
    target=TARGET,
    new_survey=NEW_SURVEY,
    bluestacks_dir=BLUESTACKS_DIR,
    preview_only=PREVIEW_ONLY
)


## 3. 사람 확인 대상 상세 조회

조사 완료 후 **사람이 직접 확인해야 하는 대상처와 확인 사유**를 조회합니다.

### 조회 대상
- **대상처 매칭 보류:** 검색 결과가 여러 개이거나 대상처를 확정하지 못한 경우
- **신규 대상처 검토:** 기존 대상처의 대체가 필요하거나 신규 후보를 확인해야 하는 경우
- **메뉴 확인:** 기존 메뉴가 사라졌거나 대체 메뉴를 검토해야 하는 경우
- **기술적 오류:** 앱 오류 등으로 조사를 정상적으로 완료하지 못한 경우

### 출력 정보
조사로그에 저장된 내용에 따라 다음 정보를 표시합니다.

| 항목 | 내용 |
|---|---|
| 기본 정보 | 지역, 엑셀 행 번호, 대상처명 |
| 조사 판정 | 상태 코드 및 사람 확인 사유 |
| 대상처 후보 | 검색된 후보 대상처 정보 |
| 신규 대상처 | 대체 대상처 추천 및 후보 정보 |
| 메뉴 정보 | 기존 메뉴, 추천 메뉴 및 후보 |
| 추가 정보 | 사업자주소, 가게주소, 비고 |

### 실행 방법
1. 조사 실행 셀이 완료되었는지 확인합니다.
2. 아래 **사람 확인 대상 조회 코드**를 실행합니다.
3. 출력된 대상처와 사유를 확인하고 필요한 항목만 수동 검토합니다.

> **주의:** 이 셀은 저장된 Excel의 `조사로그`를 읽기만 합니다. 배민 앱을 다시 조작하거나 Excel 데이터를 수정하지 않습니다.


In [ ]:
from pathlib import Path
from openpyxl import load_workbook

# 실제 조사 결과 파일 선택
LOG_FILE = Path(OUTPUT_FILE)

if MODE in ("전체", "조합"):
    LOG_FILE = LOG_FILE.with_name(LOG_FILE.stem + "_자동전체조사.xlsx")
elif MODE in ("행", "대상처") and not NEW_SURVEY:
    full_file = LOG_FILE.with_name(LOG_FILE.stem + "_자동전체조사.xlsx")
    if full_file.exists():
        LOG_FILE = full_file

book = load_workbook(LOG_FILE, read_only=True, data_only=True)

try:
    sheet = book["조사로그"]
    rows = sheet.iter_rows(values_only=True)
    headers = next(rows)
    records = [dict(zip(headers, row)) for row in rows]

    review_records = [
        r for r in records
        if "확인" in str(r.get("조사판정") or "")
        or "기술 실패" in str(r.get("조사판정") or "")
        or "REVIEW" in str(r.get("상태 코드") or "")
        or str(r.get("상태 코드") or "") in {
            "TECHNICAL_FAIL", "REPLACEMENT_NEEDED",
            "NO_REPLACEMENT_CANDIDATE", "REPLACEMENT_SEARCH_FAIL"
        }
    ]

    print("===== 사람 확인 대상 상세 조회 =====")
    print("전체 조사로그:", len(records), "건")
    print("사람 확인 대상:", len(review_records), "건")

    fields = [
        ("조사판정", "판정"),
        ("상태 코드", "상태"),
        ("확인사유", "확인 사유"),
        ("대상처 검색 후보", "검색 후보"),
        ("신규 대상처 추천", "신규 대상처 추천"),
        ("신규 대상처 후보 상세", "신규 대상처 후보"),
        ("기존 메뉴", "기존 메뉴"),
        ("메뉴 추천", "추천 메뉴"),
        ("메뉴 후보 상세", "메뉴 후보"),
        ("현재 사업자주소", "사업자주소"),
        ("현재 가게주소", "가게주소"),
        ("비고", "비고"),
    ]

    for r in review_records:
        print("\n" + "=" * 45)
        print(f"[{r.get('지역')}] {r.get('엑셀행')}행 | {r.get('대상처')}")

        for key, label in fields:
            value = r.get(key)
            if value is not None and str(value).strip():
                print(f"\n[{label}]\n{value}")

    if not review_records:
        print("사람 확인 대상이 없습니다.")

finally:
    book.close()
